## Beaty Museum Informatics x MDS Capstone: Project Proposal
#### May 8, 2026
Authors: Molly Kessler, Wendy Frankel, William Song, Johnson Chuang

Project Mentor: Payman Nickchi, UBC MDS Faculty

Capstone Partner: Paul Bucci, Informatics Curator at Beaty Biodiversity Museum

### Executive Summary

At the Beaty Biodiversity Museum, curators keep millions of physical specimen organized and labeled based on the most current scientific consensus about their taxonomy, including the species name and phylogenetic tree. Since taxonomy is constantly changing, many different synonyms may be used to label the same species in different publications (Sandall et al., 2023). We propose a web app that synthesizes taxonomic information from many online sources allowing curators to easily compare competing taxonomic information to decide which designation to incorporate into the museum's database.


### Introduction

Taxonomy refers to classifying organisms into hierarchical groups based on shared characteristics and evolutionary relationships. Within Beaty, curators are responsible for ensuring every specimen is identified, classified, and stored appropriately (Rosselló-Móra, 2011). Since taxonomy is an ever evolving field, individual species names can change over time, introducing synonyms, basionyms, variants, subspecies, and more. Beyond just the name, entire taxonomic classifications (e.g. family, order) can change, introducing further uncertainty around where organisms should physically and digitally be stored.


Many online databases compile updated species names and taxonomic classification changes from millions of individual publications. Currently, curators access these resources individually, manually comparing competing information based on their expert knowledge of Beaty's current organization scheme. Our web app will speed up this process by making API calls to online sources and creating a combined list of species synonyms and additional information — including classification, range, date of discovery, and more — to help curators decide which synonym to use. This will give curators a more consistent, reproducible, and faster workflow to classify and label specimens.

### Data & Exploratory Data Analysis (EDA)

Data sources vary by taxonomic group, with some sources containing information for all taxa (e.g. GBIF<sup>1</sup>, GenBank<sup>2</sup>) and others being more specific to a particular group (e.g. Index Fungorum<sup>3</sup> for fungi). API calls return JSON-formatted information about a species. In particular, we want to access information related to species names and taxonomy, such as species name synonyms, variants, and subspecies. The size of data varies by species — a species might have no synonyms or dozens, and any given synonym might have many variants, subspecies, etc.


In [6]:
from scripts.APIs.GBIF import get_gbif_synonyms
from scripts.APIs.IndexFungorum import get_indexfungorum_synonyms

species = "Amanita muscaria"

gbif_synonyms = set(get_gbif_synonyms(species).keys())
if_synonyms = set(get_indexfungorum_synonyms(species).keys())

all_synonyms = gbif_synonyms | if_synonyms

df = pd.DataFrame([
    {
        "Synonym": name,
        "Sources": ", ".join([
            s for s, sset in [("GBIF", gbif_synonyms), ("Index Fungorum", if_synonyms)]
            if name in sset
        ])
    }
    for name in sorted(all_synonyms)
])
df

,Synonym,Sources
0,Agaricus aureolus,"GBIF, Index Fungorum"
1,Agaricus imperialis,"GBIF, Index Fungorum"
2,Agaricus muscarius,"GBIF, Index Fungorum"
3,Agaricus nobilis,"GBIF, Index Fungorum"
4,Agaricus pseudoaurantiacus,"GBIF, Index Fungorum"
5,Agaricus puellus,"GBIF, Index Fungorum"
6,Amanita aureola,"GBIF, Index Fungorum"
7,Amanita circinnata,"GBIF, Index Fungorum"
8,Amanita formosa,"GBIF, Index Fungorum"
9,Amanita muscaria,"GBIF, Index Fungorum"


Table 1. Species *Amanita muscaria* synonyms from GBIF and Index Fungorum. 

A main challenge when working with this data is that performing many API calls will become very slow, and we will need to optimize our code and likely implement caching to speed up searches. Additionally, the output format will vary across sources, requiring substantial data cleaning and post-processing to collate data into a consistent format.

### Data Science Approach

Our data pipeline will automate the collection of records from online sources, displaying them in a visual interface for curators to make faster and more informed decisions. We will provide search results from authoritative sources for all kinds of taxa (i.e. fungi, insects, etc) in a consistent format by implementing API calls to each source and unifying the output format. Each source will have its own script responsible for retrieving relevant records and cleaning them into the consistent format. To ensure consistency as the codebase grows, we will consider defining a Python abstract class with database-specific subclasses so that future developers can easily add new sources that conform to the same contract.
  
A key part of what we are building is not just data retrieval, but also incorporating provenance and context. Authorities change over time as data sources become stale, funding lapses, and scientific consensus shifts. Curators don't just need a list of synonyms — they need to know what sources list a name, when it was published, and how widely it is recognized across different scientific communities. For example, a well established synonym that is primarily used in South America may be less relevant for Beaty than a less common synonym that is most used in Canada. Metrics such as coverage (how many databases list the name), recency (how recently the name was published/became popular), specificity (whether the name is common in more general vs. taxa-specific resources), and more will give curators a quick sense of which names are recognized and current.
 
We will explore a variety of interface designs to present the reconciled information to curators, including a comparison table, a node-based expandable graph, and an LLM summary view powered by a Retrieval-Augmented Generation<sup>4</sup> (RAG) approach. In all designs, our priority is to surface information without introducing bias, and always to provide information to curators, not to make any automated decisions. Aggregated summaries in particular carry a risk of obscuring disagreement between sources, so we will be thoughtful about how we present confidence and provenance alongside any LLM summary. 

The final deliverable will be a working pipeline and app hosted on Beaty's GitHub repository, with documentation that allows curators and future developers to use and extend it. A curator will be able to access the app at a public URl and use it to search a species name and retrieve a reconciled view of relevant records from all configured sources, along with provenance metadata for each result.

### Evaluation & Success Criteria

We define success in terms of the reliability and completeness of the information we surface, and whether that information helps curators make faster decisions and feel more informed. Our primary success condition is that the pipeline can accurately retrieve, normalize, and compare records from the configured data sources, and present them to curators in a way that is clear, transparent, and free of systematic bias.

We will evaluate success through feedback from curators at Beaty. Their ability to use the tool effectively, and their confidence in the provenance and authority of the information it surfaces, is our most important qualitative measure of success. If curators can understand where each piece of data came from and feel the tool has improved their ability to make informed determinations, we will have succeeded. In order to achieve success for curators working across many different collections, we will need to implement many different sources. Currently we expect to implement over 30 individual sources into our pipeline (see Appendix A), but this number may grow as we encourage curators to request the sources they use. Time permitting, we would also like to conduct a structured comparison in which curators process a new specimen submission using their current workflow and then again using our tool, measuring both the time taken and the agreement between the resulting taxonomic labels.

We will be extremely attentive to evaluating the LLM component for bias, hallucinations, and appropriate level of detail. We will review LLM outputs across a range of taxa and source combinations to check whether the model consistently favors certain databases, naming conventions, or geographic authorities without justification from the underlying data. If we identify issues, we will attempt to address them through prompt engineering or model selection, and if unsuccessful, will carefully consider whether it is safe and ethical to include an LLM at this time.

### Timeline

**Before May 8**: Present proposal and initial prototypes to mentor and capstone partner for feedback.

**May 15 – 22**: Develop preliminary search system capable of returning species names collated from 10+ API sources. Gather curator feedback on working and paper prototypes.

**By May 29**: Add additional species details and metrics (coverage, recency, etc.) for each synonym. Complete initial LLM development and begin evaluation. Continue adding APIs.

**June 8**: Finish implementing 30+ APIs. Complete initial model evaluation. Submit a runnable draft to mentor for review. 

**June 8 – 10**: Finalize presentation slides.

**June 24 – 25**: Deliver final version of report and data product to capstone partner. Present final product.


### References

1. GBIF: The Global Biodiversity Information Facility. 2026, www.gbif.org/what-is-gbif. Accessed 13 Jan. 2020.

2. GenBank Overview. National Center for Biotechnology Information, National Library of Medicine, www.ncbi.nlm.nih.gov/genbank/. Accessed 7 May 2026.

3. Index Fungorum. The Royal Botanic Gardens, Kew, 2025, www.indexfungorum.org. Accessed 6 May 2026.

4. Retrieval-Augmented Generation. Wikipedia, Wikimedia Foundation, en.wikipedia.org/wiki/Retrieval-augmented_generation. Accessed 7 May 2026.

5. Rosselló-Móra, R. "Taxonomy." Encyclopedia of Astrobiology, edited by M. Gargaud et al., Springer, 2011, doi.org/10.1007/978-3-642-11274-4_1562.

6. Sandall, Emily L., et al. "A Globally Integrated Structure of Taxonomy to Support Biodiversity Science and Conservation." Trends in Ecology & Evolution, vol. 38, no. 12, 2023, pp. 1143–1153, doi.org/10.1016/j.tree.2023.08.004.

7. Sayers, Eric W., et al. "GenBank." Nucleic Acids Research, vol. 48, no. D1, 8 Jan. 2020, pp. D84–D86, doi.org/10.1093/nar/gkz956.


### Appendix A: Data Sources

The following table lists the individual data sources expected to be integrated into our pipeline, along with their primary taxonomic scope.

| Source | Taxonomic Scope |
|--------|-----------------|
| [AlgaeBase](https://www.algaebase.org/) | Global algal taxonomy |
| [AntWeb](https://antweb.org/) | Ant taxonomy |
| [Bryophyte Portal](https://bryophyteportal.org/portal/) | Mosses, liverworts, hornworts |
| [Catalogue of Life](https://www.catalogueoflife.org/) | Global species checklist across all kingdoms |
| [CCH2](https://cch2.org/portal/index.php) | Plant species & herbarium collections |
| [Darwin Core (TDWG)](https://dwc.tdwg.org/) | Biodiversity data standard (terms for species, specimens, occurrences) |
| [FishBase](https://www.fishbase.se/search.php) | Fish / marine life |
| [GBIF Occurrence Portal](https://www.gbif.org/occurrence/search) | Global biodiversity occurrences |
| [GenBank](https://www.ncbi.nlm.nih.gov/genbank/) | Nucleotide sequences across all life (genomic DNA, mRNA, rRNA, viral genomes, metagenomes) |
| [iDigBio Portal](https://portal.idigbio.org/portal/search) | Aggregated US biodiversity data |
| [iNaturalist](https://www.inaturalist.org/) | Plants, animals, fungi (citizen science) |
| [Index Fungorum](https://www.indexfungorum.org/) | Scientific names of fungi |
| [Lichen Portal](https://lichenportal.org/portal/) | Lichenized fungi |
| [Macroalgae.org](https://macroalgae.org/portal/) | Algae herbarium data |
| [Mid-Atlantic Herbaria](https://midatlanticherbaria.org/portal/) | Mid-Atlantic plant collections |
| [Mushroom Observer](https://mushroomobserver.org/) | Mushrooms / fungi |
| [MycoMap](https://www.mycomap.com/) | Fungal observations & analytics |
| [MyCoPortal](https://mycoportal.org/portal/) | Fungi / fungal collections |
| [NANSH](https://nansh.org/portal/index.php) | Small Herbaria (plants) |
| [NE Herbaria Portal](https://portal.neherbaria.org/portal/) | Northeastern US herbarium collections |
| [PteridoPortal](https://pteridoportal.org/portal/) | Pteridophytes (ferns & allies) |
| [SERNEC](https://sernecportal.org/portal/index.php) | 233 herbaria in southeastern USA |
| [SW Biodiversity](https://swbiodiversity.org/) | Botanical data (Arizona–New Mexico) |
| [Symbiota Portals](https://symbiota.org/symbiota-portals/) | Many taxonomic groups depending on portal |
| [Tropicos](https://www.tropicos.org/home) | Plants / herbs |
| [VertNet](https://vertnet.org/) | Vertebrate biodiversity data |